## Classification (분류)

### Binary Classification (이진 분류)
----
* 둘 중 하나를 결정하는 문제
* 합격/불합격, 스팸메일인지/아닌지
* 관심 있는 (맞추고자 하는) class를 양성이라고 함 (⚠️주의: 양성이 좋은 경우는 아님)
    - 스팸메일을 분류하는 문제의 경우 스팸메일이 양성

### Logistic Regression (로지스틱 회귀)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## 60이상이면 합격하는 이진분류
score = np.array([45, 50, 55, 60, 65, 70])
result = np.array([0, 0, 0, 1, 1, 1])

plt.plot(score, result, "o-")

이 함수는 선형 회귀로 표현하기 어려움. 따라서 위와 같은 함수를 잘 맞추는 함수 $f(wx+b)$가 있다고 가정

위 그래프의 $y$축 값은 $p(y=1|x)$로 해석 가능.

**IDEA**: $p(y=1|x)=f(wx+b)$를 만족하는 함수 $f$를 찾자.

$P=p(y=1|x)$라 하면, $P\in (0,1)$. ($P=0, 1$에서는 아래의 logit이 정의되지 않으므로 양 끝은 제외)

$wx+b$는 모든 실수 값을 가질 수 있으므로, $f^{-1}(P)=wx+b$는 $(-\infty,\infty)$ 범위를 만족해야함

* Odds: $\frac{P}{1-P} \in (0,\infty)$
* logit function: $\operatorname{logit}(P)=\log\frac{P}{1-P}  \in  (-\infty,\infty)$

**로지스틱 회귀 전개**

$z:=wx+b$

$\log\frac{P}{1-P} = z$

$\log\frac{1-P}{P}=-z$

$\frac{1-P}{P}=e^{-z}$ (양변에 지수 함수 적용)

$\frac{1}{P}-1=e^{-z}$

$\frac{1}{P}=1+e^{-z}$

$P=\frac{1}{1+e^{-z}}$ : 로지스틱 함수, $\sigma(z)$. (S자 모양 함수를 통칭하는 시그모이드(sigmoid) 함수 중 하나이며, 머신러닝에서는 보통 시그모이드 함수라고 부름)

$P=\frac{1}{1+e^{-(wx+b)}}$

**임계값 설정**: 예측 확률 $P$를 클래스(0 또는 1)로 변환
$$
\text{class} =
\begin{cases}
1 & \text{if } P \geq 0.5 \\
0 & \text{otherwise}
\end{cases}
$$

* $P \geq 0.5 \iff e^{-z} \leq 1 \iff z = wx+b \geq 0$
* 즉, $wx+b$의 부호로 분류가 결정되며 $wx+b=0$이 **결정 경계(decision boundary)**


In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

z = np.arange(-7, 7, 0.1)
sigma_z = sigmoid(z)

plt.plot(z, sigma_z)
plt.axvline(0.0, color='k')
plt.ylim(-0.1, 1.1)
plt.xlabel('z')
plt.ylabel('$\sigma (z)$')

# y 축의 눈금과 격자선
plt.yticks([0.0, 0.5, 1.0])
ax = plt.gca()
ax.yaxis.grid(True)

plt.tight_layout()
#plt.savefig('figures/03_02.png', dpi=300)
plt.show()

#### 선형 회귀와의 관계

* 로지스틱 회귀 = **선형 회귀의 출력에 시그모이드 함수를 씌운 것**

| | 선형 회귀 (4장) | 로지스틱 회귀 |
|---|---|---|
| 선형 결합 | $z = X\mathbf{w} + b$ | $z = X\mathbf{w} + b$ |
| 출력 | $\hat{y} = z$ (실수) | $\hat{y} = \sigma(z)$ (0~1 사이 확률) |
| 비용 함수 | MSE | 교차 엔트로피 (Cross Entropy) |

#### 비용 함수: 왜 MSE를 쓰지 않을까?

시그모이드 출력에 MSE를 사용하면 비용 함수가 볼록(convex)하지 않은 모양이 되어, 경사하강법이 **지역 최솟값(local minimum)** 에 빠질 수 있음

In [ ]:
from IPython.display import Image
Image(url='https://raw.githubusercontent.com/Hyelim-Shin/AIdev/master/Images/10-1.sigmoid_mse.png', width=300)

시그모이드 함수의 출력값은 0에서 1사이. 즉, 실제값이 1일 때 예측값이 0에 가까와지면 오차 증가, 실제값이 0일 때 예측값이 1에 가까워지면 오차가 커져야 함.

$$
if\ y=1 \rightarrow L(\hat{y},y)=-log(\hat{y})
$$

$$
if\ y=0 \rightarrow L(\hat{y},y)=-log(1-\hat{y})
$$

In [ ]:
Image(url='https://raw.githubusercontent.com/Hyelim-Shin/AIdev/master/Images/10-1.cross_entropy.png', width=300)

하나의 식으로 통합: **이진 교차 엔트로피 (Binary Cross Entropy, BCE)**

$$
L(\mathbf{w},b)=-\frac{1}{n} \sum_{i=1}^n [y^{(i)} \log\hat{y}^{(i)} + (1-y^{(i)}) \log(1-\hat{y}^{(i)})]
$$

* $y^{(i)}$: 실제 정답 (0 또는 1), $\hat{y}^{(i)}=\sigma(z^{(i)})$: 모델이 예측한 확률 $P(y=1|x^{(i)})$
* $y^{(i)}=1$이면 두 번째 항이 0 → $-\log\hat{y}^{(i)}$만 남음
* $y^{(i)}=0$이면 첫 번째 항이 0 → $-\log(1-\hat{y}^{(i)})$만 남음

**기울기 유도**

$z = \mathbf{w}^T\mathbf{x} + b$, $\hat{y}=\sigma(z)$이므로 연쇄 법칙(chain rule)을 이용 (편의상 샘플 하나의 손실 $\ell$에 대해 계산, 위첨자 $(i)$ 생략)

$$
\frac{\partial \ell}{\partial \mathbf{w}} = \frac{\partial \ell}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} \cdot \frac{\partial z}{\partial \mathbf{w}}
$$

**Step 1. 시그모이드 함수의 미분**

$$
\sigma(z) = (1+e^{-z})^{-1}
\quad\Rightarrow\quad
\sigma'(z) = \frac{e^{-z}}{(1+e^{-z})^2} = \frac{1}{1+e^{-z}} \cdot \frac{e^{-z}}{1+e^{-z}} = \sigma(z)\big(1-\sigma(z)\big)
$$

즉, $\frac{\partial \hat{y}}{\partial z} = \hat{y}(1-\hat{y})$

**Step 2. 손실 함수를 $\hat{y}$로 미분**

$$
\ell = -[y \log\hat{y} + (1-y) \log(1-\hat{y})]
\quad\Rightarrow\quad
\frac{\partial \ell}{\partial \hat{y}} = -\frac{y}{\hat{y}} + \frac{1-y}{1-\hat{y}} = \frac{\hat{y}-y}{\hat{y}(1-\hat{y})}
$$

**Step 3. 두 결과를 곱하기**

$$
\frac{\partial \ell}{\partial z} = \frac{\hat{y}-y}{\hat{y}(1-\hat{y})} \cdot \hat{y}(1-\hat{y}) = \hat{y}-y
$$

* 시그모이드의 미분 $\hat{y}(1-\hat{y})$이 약분되어 매우 간단한 형태가 됨

**Step 4. $\mathbf{w}$, $b$에 대한 기울기**

$\frac{\partial z}{\partial \mathbf{w}} = \mathbf{x}$, $\frac{\partial z}{\partial b} = 1$이므로, $n$개 샘플에 대해 평균을 내면

<center>

$\frac{\partial L}{\partial \mathbf{w}} = \frac{1}{n}\sum_{i=1}^{n} (\hat{y}^{(i)}-y^{(i)})\,\mathbf{x}^{(i)} = \frac{1}{n} X^T(\hat{\mathbf{y}}-\mathbf{y})$

$\frac{\partial L}{\partial b} = \frac{1}{n}\sum_{i=1}^{n} (\hat{y}^{(i)}-y^{(i)})$

</center>

**선형 회귀(MSE)와 비교**

* 4-2의 MSE 기울기 $\frac{\partial L}{\partial w} = \frac{2}{n}\sum_{i=1}^{n} (\hat{y}^{(i)}-y^{(i)})\, x^{(i)}$와 상수 2를 제외하면 **같은 모양** (단, $\hat{y}$가 $wx+b$가 아니라 $\sigma(wx+b)$)
* 따라서 4-2의 경사하강법 코드에서 **예측 함수만 $\sigma(X\mathbf{w}+b)$로 바꾸면** 그대로 사용 가능

#### 분류 모델의 평가

**정확도 (Accuracy)** $= \frac{TP+TN}{TP+TN+FP+FN}$: 전체 샘플 중 맞게 예측한 샘플의 비율

* 한계: 클래스 비율이 불균형한 데이터에서는 정확도만으로 모델을 평가하기 어려움
    - 예: 전체 메일 중 스팸메일이 1%뿐이라면, 모든 메일을 '정상'으로 예측해도 정확도 99%

**혼동 행렬 (Confusion Matrix)**

| | 예측: 0 (Negative) | 예측: 1 (Positive) |
|---|---|---|
| **실제: 0** | TN (True Negative) | FP (False Positive) |
| **실제: 1** | FN (False Negative) | TP (True Positive) |

* 두 번째 글자(P/N): 모델이 예측한 클래스
* 첫 번째 글자(T/F): 그 예측이 맞았는지 여부

**정밀도, 재현율, F1 점수**

* **정밀도 (Precision)** $= \frac{TP}{TP+FP}$: 양성으로 예측한 것 중 실제 양성의 비율
    - FP를 줄이는 것이 중요할 때 (예: 스팸메일 분류 - 정상 메일을 스팸으로 분류하면 중요한 메일을 놓침)
* **재현율 (Recall)** $= \frac{TP}{TP+FN}$: 실제 양성 중 양성으로 맞게 예측한 비율
    - FN을 줄이는 것이 중요할 때 (예: 암 진단 - 실제 환자를 놓치는 것이 더 위험)
* **F1 점수** $= 2 \times \frac{Precision \times Recall}{Precision + Recall}$: 정밀도와 재현율의 조화 평균

**임계값과 정밀도/재현율의 관계 (trade-off)**

* 임계값을 높이면: 확실한 경우만 양성으로 예측 → 일반적으로 정밀도 ↑, 재현율 ↓
* 임계값을 낮추면: 양성 예측이 많아짐 → 일반적으로 정밀도 ↓, 재현율 ↑

#### 실습: 타이타닉 생존 예측

2-5에서 사용한 타이타닉 데이터로 승객의 생존 여부(`survived`: 1=생존, 0=사망)를 예측

In [ ]:
import pandas as pd
import seaborn as sns

titanic = sns.load_dataset('titanic')
titanic.head()

##### 1) 데이터 전처리 (3-1 복습)

* 사용할 특성: `pclass`(객실 등급), `sex`(성별), `age`(나이), `sibsp`(형제/배우자 수), `parch`(부모/자녀 수), `fare`(요금), `embarked`(탑승 항구)

In [ ]:
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df = titanic[features + ['survived']].copy()
df.isnull().sum()

In [ ]:
# 누락된 값 대체
df['age'] = df['age'].fillna(df['age'].median())                    # 나이: 중앙값
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])    # 탑승 항구: 최빈값

# 범주형 데이터 인코딩
df['sex'] = df['sex'].map({'male': 0, 'female': 1})                 # 매핑
df = pd.get_dummies(df, columns=['embarked'], drop_first=True, dtype=float)  # 원-핫 인코딩

df.head()

In [ ]:
X = df.drop(columns='survived').values.astype(float)
y = df['survived'].values
feature_names = df.drop(columns='survived').columns

print(X.shape, y.shape)
print('생존 비율:', y.mean())

##### 2) 데이터 분할 및 표준화

In [ ]:
test_df = df.sample(frac=0.2, random_state=42)   # 20%를 무작위로 뽑아 테스트 데이터로 사용
train_df = df.drop(test_df.index)                # 나머지 80%는 훈련 데이터로 사용

X_train, y_train = train_df.drop(columns='survived').values.astype(float), train_df['survived'].values
X_test, y_test = test_df.drop(columns='survived').values.astype(float), test_df['survived'].values

# train 데이터로 평균과 표준편차 계산 후, test 데이터에도 같은 값으로 변환
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train_sc = (X_train - mean) / std
X_test_sc = (X_test - mean) / std

##### 3) 모델 학습 (경사하강법)

In [ ]:
def predict_proba(X, w, b):
    # TODO: 선형 결합(np.dot(X, w) + b)에 sigmoid 함수를 씌운 값을 반환
    return None

def bce(y, p):
    eps = 1e-7                        # log(0)을 방지하기 위한 작은 값
    p = np.clip(p, eps, 1 - eps)
    # TODO: BCE 식으로 loss 계산 (np.mean, np.log 사용)
    return None

def train_logistic(X, y, lr, n_epochs):
    n, m = X.shape
    w = np.zeros(m)
    b = 0.0
    loss_history = []
    for epoch in range(n_epochs):
        # TODO: 1) 예측 (predict_proba 사용)
        p = None
        # TODO: 2) loss 계산 후 loss_history에 추가 (bce 사용)

        # TODO: 3) 기울기 계산 (dw = np.dot(X.T, p - y) / n, db = (p - y)의 평균)
        #       X.T: X의 전치 행렬 (행과 열을 바꾼 행렬)
        dw = None
        db = None
        # TODO: 4) w, b 업데이트 (학습률 lr 사용)

    return w, b, loss_history

In [ ]:
w, b, loss_history = train_logistic(X_train_sc, y_train, lr=0.1, n_epochs=1000)

plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('BCE loss')
plt.show()

In [ ]:
# 학습된 가중치: 양수이면 생존 확률을 높이고, 음수이면 낮추는 특성
pd.Series(w, index=feature_names).sort_values()

* `sex`(여성=1)는 생존 확률을 크게 높이고, `pclass`(등급 숫자가 클수록 낮은 등급)는 생존 확률을 낮춤

##### 4) 예측

In [ ]:
# 생존할 확률 P(y=1|x)
predict_proba(X_test_sc[:3, :], w, b)

In [ ]:
def predict(X, w, b, threshold=0.5):
    return (predict_proba(X, w, b) >= threshold).astype(int)   # 임계값 0.5 이상이면 1

predict(X_test_sc[:3, :], w, b)

In [ ]:
y_test[:3]

##### 5) 평가

앞의 **분류 모델의 평가**에서 배운 척도를 직접 계산

**정확도 (Accuracy)**

In [ ]:
y_pred = predict(X_test_sc, w, b)

# TODO: 정확도 계산 (y_pred와 y_test가 같은 샘플의 비율)
accuracy = None
print(f'Accuracy: {accuracy:.3f}')

**혼동 행렬 (Confusion Matrix)**

In [ ]:
# 예측도 1, 실제도 1인 샘플의 개수
TP = np.sum((y_pred == 1) & (y_test == 1))

# TODO: TP와 같은 방법으로 TN, FP, FN 계산
TN = None
FP = None
FN = None

confusion = pd.DataFrame([[TN, FP], [FN, TP]],
                         index=['실제 0', '실제 1'], columns=['예측 0', '예측 1'])
confusion

**정밀도, 재현율, F1 점수**

In [ ]:
# TODO: TP, TN, FP, FN을 이용해 정밀도, 재현율, F1 점수 계산
precision = None
recall = None
f1 = None

print(f'Precision: {precision:.3f}')
print(f'Recall   : {recall:.3f}')
print(f'F1 score : {f1:.3f}')

In [ ]:
## 실습
# 1) 임계값(threshold)을 0.3, 0.7로 바꾸었을 때 Precision과 Recall이 어떻게 변하는지 확인하세요.
# 2) 표준화를 하지 않은 X_train, X_test로 학습하면 결과가 어떻게 달라지는지 확인하세요.